# Medical Knowledge Distillation Demo (Colab)

End-to-end pipeline: download models, prepare data, distill teacher knowledge into Qwen student, compare QA outputs, and visualize results.

**Requirements:** Hugging Face token for model downloads. Dataset: [`HoangHa/medical-data`](https://huggingface.co/datasets/HoangHa/medical-data) (`RandomQA` subset). GPU recommended (H100 ideal).

> Outputs are for research only — not medical advice.

In [ ]:
# Optional: mount Google Drive for persistent checkpoints
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import os
import subprocess
from pathlib import Path

# Clone or use uploaded project
PROJECT_DIR = Path('/content/KD_SLM')
if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/abinesha312/KD_SLM.git', str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', '.'], check=True)
print('Project ready at', PROJECT_DIR)

In [ ]:
# Set Hugging Face token
import os

HF_TOKEN = ''  # <-- paste your token here
# Or from Colab secrets:
# from google.colab import userdata
# HF_TOKEN = userdata.get('HF_TOKEN')

os.environ['HF_TOKEN'] = HF_TOKEN
assert HF_TOKEN and HF_TOKEN != 'your_token_here', 'Set HF_TOKEN before continuing'

In [ ]:
# GPU check
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!python scripts/01_download_models.py

In [ ]:
!python scripts/02_prepare_dataset.py

In [ ]:
!python scripts/03_generate_teacher_outputs.py --split all

In [ ]:
!python scripts/04_distill_train.py

In [ ]:
!python scripts/05_run_qa_comparison.py --split test --max-samples 100

In [ ]:
!python scripts/06_visualize_results.py

In [ ]:
from IPython.display import Image, display
import pandas as pd
from pathlib import Path

figures = Path('outputs/figures')
for img in sorted(figures.glob('*.png')):
    print(img.name)
    display(Image(filename=str(img)))

comparison = pd.read_csv('outputs/qa_results/comparison.csv')
display(comparison.groupby('model_name')[['latency_sec', 'tokens_per_sec', 'peak_vram_gb']].mean())
comparison.head(6)